In [1]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

/home/doaa/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-09 14:47:49.495706: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-09 14:47:50.036898: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-09 14:47:53.786744: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different n

In [2]:
model_name = "Qwen/Qwen1.5-0.5B-Chat"


In [70]:
# ===== Load Model (4bit) =====
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="cpu",
    low_cpu_mem_usage=True,
    torch_dtype="auto",
)

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# ===== Load Dataset =====
dataset = load_dataset("json", data_files="train.json")

def clean(example):
    for field in ["instruction", "input", "output"]:
        value = example[field]

        if isinstance(value, list):
            value = " ".join(str(v) for v in value if isinstance(v, str))

        if isinstance(value, (int, float)):
            value = str(value)

        if value is None:
            value = ""

        example[field] = value
    return example

def check_tokens(example):
    ids = example["input_ids"]
    vocab_size = tokenizer.vocab_size
    if max(ids) >= vocab_size:
        print("⚠ BAD SAMPLE FOUND:", max(ids))
    return example


dataset = dataset.map(clean)

# ===== Format Data =====
def format(example):
    prompt = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n"
    )

    example["text"] = prompt + example["output"]
    return example


dataset = dataset.map(format)


In [ ]:
# ===== Tokenization =====
def tokenize(example):
    # Tokenize the text
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    # Duplicate input_ids to labels for Causal LM training
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

# Get the list of columns to remove (instruction, input, output, text)
column_names = dataset["train"].column_names

# Apply tokenization and REMOVE the text columns immediately
dataset = dataset.map(
    tokenize, 
    batched=True, 
    remove_columns=column_names
)

print("Columns remaining:", dataset["train"].column_names)
# Should only print: ['input_ids', 'attention_mask', 'labels']

Map: 100%|██████████| 69/69 [00:00<00:00, 1089.82 examples/s]

Map: 100%|██████████| 69/69 [00:00<00:00, 1089.82 examples/s]

Columns remaining: ['input_ids', 'attention_mask', 'labels']


Map: 100%|██████████| 69/69 [00:00<00:00, 1089.82 examples/s]

Columns remaining: ['input_ids', 'attention_mask', 'labels']


In [73]:
# ===== LoRA Config =====
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)


In [74]:
# ===== Data Collator =====
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt"
)


In [ ]:
# ===== Training =====
training_args = TrainingArguments(
    output_dir="./finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=200,
    fp16=False,
    use_cpu=True,
    remove_unused_columns=False,
)

In [76]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    data_collator=data_collator,
)

In [77]:
print(dataset["train"][0])


{'input_ids': [14374, 29051, 510, 2610, 525, 364, 1162, 37, 7671, 354, 516, 264, 11657, 714, 49665, 2613, 4128, 1614, 13, 4615, 5306, 6234, 374, 69956, 6859, 40209, 323, 279, 22707, 3552, 374, 50235, 34351, 62, 16, 17, 18, 13, 3155, 537, 16400, 1493, 23594, 7241, 458, 7600, 353, 376, 5788, 9, 498, 1119, 18484, 1105, 382, 14374, 5571, 510, 9707, 11, 1246, 525, 498, 3351, 1939, 14374, 5949, 510, 40, 1079, 264, 2613, 6236, 6331, 6941, 356, 10808, 7671, 354, 11, 323, 358, 1079, 30201, 1632, 0, 2585, 646, 358, 7789, 498, 448, 4285, 9079, 30, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 1

In [ ]:
trainer.train()


Step,Training Loss


In [ ]:
# ===== Save =====
model.save_pretrained("finetuned_lora")
tokenizer.save_pretrained("finetuned_lora")
